# What does a prompt template do to the component interactions? — ImageNet, ViT-B-16 & ViT-L-14

The zero-shot text side can be written several ways for the same class $c$:

* $\mathbf B'(c)$ — the **bare class name** (`"tench"`), the reference, and
* $\mathbf B(c)$ — a **templated prompt** (`"an image of tench"`, `"a photo of a tench"`),

each decomposed by the text-tower PRS into $K_t$ component rows that **sum exactly to the
$L_2$-normalised text embedding** (verified against a real `encode_text` forward: max abs err $\sim10^{-6}$,
$\cos = 1.000000$). The image side $\mathbf A(i)$ is never touched.

The interaction matrix is
$$M(i,c) \;=\; \mathbf B(c)\,\mathbf A(i)^{\!\top} \in \mathbb R^{K_t\times K_v},
\qquad \textstyle\sum_{ab} M_{ab} \;=\; \cos\big(z_T(c),\,z_I(i)\big),$$
so **every entry is an additive share of the zero-shot score**. Writing $M'$ for the bare prompt,
$$\Delta M \;=\; M - M' \;=\; \Delta\mathbf B(c)\,\mathbf A(i)^{\!\top},
\qquad \Delta \mathbf B = \mathbf B - \mathbf B' ,$$
is an exact, additive attribution of the entire prompt effect onto (text component × vision component)
pairs. This notebook asks of $\Delta M$:

1. **§1** How much does each template move the accuracy, and *where can that movement possibly live*?
   (a two-line algebraic fact that removes most of $\Delta M$ from contention)
2. **§2** On the text side — is it *one* component that fires, or many? Gradual or concentrated?
3. **§3** $M$, $M'$, $\Delta M$ as matrices (per class and class-averaged); how concentrated is $\Delta M$
   over pairs, and which pairs carry the **class-discriminative** part $G_{ab}$.
4. **§4** The §8-style (cos, weight) plane, one series per prompt, for positive **and** negative pairs:
   do the top interactions stay the same, and who moves in *direction* vs in *magnitude*?
5. **§5** The causal test — swap components in one at a time / cumulatively and watch zero-shot accuracy.
   *This* is what answers "which component drives the change".
6. **§6** Geometry: does templating move the text embeddings **towards the images** on average, and does
   that average motion have anything to do with the accuracy?

Everything runs for **ViT-B-16** and **ViT-L-14** on **imagenet** (5 000 images, 5/class, seed 69).

In [1]:
# Thread caps must be set BEFORE numpy import.
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(v, "4")

import json
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

OUT_DIR = "output_dir/activations_and_datasets_idxs_69"
SEED    = 69
DATASET = "imagenet"
MODELS  = ["ViT-B-16", "ViT-L-14"]
BARE    = "imagenet_classnames"                     # B'  ->  "tench"
# Templated variants: label -> text-set name (fixed order; colour is assigned by identity, never rank).
# The first two differ by ONE token, the indefinite article, and land on opposite sides of the bare
# baseline (-2.0 vs +1.3 top-1 on ViT-B-16) -- that contrast is the sharpest probe in the notebook.
VARIANTS = {"an image of {}":   "imagenet_classnames_animage",
            "an image of a {}": "imagenet_classnames_animagea",
            "a photo of a {}":  "imagenet_classnames_aphoto"}
PROMPTS = ["bare"] + list(VARIANTS)

device  = "cuda:0" if torch.cuda.is_available() else "cpu"
FIG_DIR = f"{OUT_DIR}/prompt_interaction_figs"
os.makedirs(FIG_DIR, exist_ok=True)
RESULTS = {}                                        # every headline number also lands in results.json

def save_fig(fig, name, dpi=200):
    path = f"{FIG_DIR}/{name}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print("saved ->", path)
    return path

def slug(s):
    return s.replace(" ", "-").replace("{}", "X").replace("/", "-")

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 8.5, "axes.titlesize": 9})

# dataviz: categorical hues assigned by identity, in fixed order. Both sets pass the six checks on a
# white surface under --pairs all (prompt triple: worst 29.0 normal / 12.6 deutan; family pair: 31.9 /
# 25.6). Marker shape repeats the identity so it is never carried by colour alone.
P_COLOR  = dict(zip(PROMPTS, ["#2a78d6", "#eb6834", "#1baf7a", "#b3202b"]))
P_MARKER = dict(zip(PROMPTS, ["o", "X", "^", "s"]))
F_COLOR  = {"attn": "#2a78d6", "mlp": "#b3202b"}    # component family
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f2f6fc", "#2a78d6", "#123763"])
DIV = LinearSegmentedColormap.from_list("div_bo",
      ["#123763", "#2a78d6", "#f4f4f2", "#eb6834", "#7a2f10"])   # two hues + neutral midpoint

def div_im(ax, X, q=99.5):
    """Signed heatmap. The scale is a PERCENTILE, not the max: a handful of MLP entries are ~20x the
    next largest, and a max-based scale renders every other entry as blank surface."""
    v = float(np.percentile(np.abs(X), q)) or float(np.abs(X).max()) or 1e-12
    im = ax.imshow(np.clip(X, -v, v), aspect="auto", cmap=DIV,
                   norm=TwoSlopeNorm(0, -v, v), interpolation="nearest")
    ax.grid(False)
    return im, v

print("device:", device, "| models:", MODELS, "| prompts:", PROMPTS)

/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


device: cpu | models: ['ViT-B-16', 'ViT-L-14'] | prompts: ['bare', 'an image of {}', 'an image of a {}', 'a photo of a {}']


In [2]:
# Goal:  loaders. A = image components [N,Kv,d]; B["bare"], B[variant] = text comps [C,Kt,d].
# Does:  the saved PRS activations are already divided by ||output||, so sum(axis=1) IS the unit
#        embedding and every inner product below is literally a share of the cosine.

def tpath(name, model, what):
    return f"{OUT_DIR}/{name}_{what}_text_{model}_seed_{SEED}.npy"

def comp_index(L, H, M):
    """Component metadata for one tower: attn heads (layer-major) then MLP slots."""
    kind  = np.array(["attn"] * (L * H) + ["mlp"] * M)
    layer = np.array([l for l in range(L) for _ in range(H)] + list(range(M)))
    head  = np.array([h for _ in range(L) for h in range(H)] + [-1] * M)
    return kind, layer, head

def comp_names(kind, layer, head):
    return np.array([f"a{l}.{h}" if k == "attn" else f"m{l}" for k, l, h in zip(kind, layer, head)])

def stack(attn, mlp):
    """[P,L,H,d] + [P,M,d] -> [P, L*H+M, d]; the sum over axis 1 IS the embedding."""
    P, d = attn.shape[0], attn.shape[-1]
    return torch.cat([attn.reshape(P, -1, d), mlp], 1)

def _t(path):
    return torch.from_numpy(np.load(path).astype(np.float32)).to(device)

def load(model):
    A = stack(_t(f"{OUT_DIR}/{DATASET}_attn_{model}_seed_{SEED}.npy"),
              _t(f"{OUT_DIR}/{DATASET}_mlp_{model}_seed_{SEED}.npy"))
    lbl_v = np.load(f"{OUT_DIR}/{DATASET}_labels_{model}_seed_{SEED}.npy")
    B, lbl_t = {}, None
    for lab, name in [("bare", BARE)] + list(VARIANTS.items()):
        B[lab] = stack(_t(tpath(name, model, "attn")), _t(tpath(name, model, "mlp")))
        l = np.load(tpath(name, model, "labels"))
        assert lbl_t is None or (l == lbl_t).all(), "prompt sets must share the class order"
        lbl_t = l
    av = np.load(f"{OUT_DIR}/{DATASET}_attn_{model}_seed_{SEED}.npy", mmap_mode="r")
    mv = np.load(f"{OUT_DIR}/{DATASET}_mlp_{model}_seed_{SEED}.npy", mmap_mode="r")
    at = np.load(tpath(BARE, model, "attn"), mmap_mode="r")
    mt = np.load(tpath(BARE, model, "mlp"), mmap_mode="r")
    kv = comp_index(av.shape[1], av.shape[2], mv.shape[1])
    kt = comp_index(at.shape[1], at.shape[2], mt.shape[1])
    return dict(A=A, B=B, lbl_v=lbl_v, lbl_t=lbl_t, kv=kv, kt=kt,
                nv=comp_names(*kv), nt=comp_names(*kt),
                n_attn_v=int(av.shape[1] * av.shape[2]), n_attn_t=int(at.shape[1] * at.shape[2]))

D = {m: load(m) for m in MODELS}
for m in MODELS:
    d = D[m]
    print(f"{m:9s} A {tuple(d['A'].shape)}   B {tuple(d['B']['bare'].shape)}  x {len(d['B'])} prompts")
    print(f"{'':9s} ||sum of components||  image {d['A'].sum(1).norm(dim=1).mean():.6f}   " +
          "   ".join(f"{p} {d['B'][p].sum(1).norm(dim=1).mean():.6f}" for p in PROMPTS) + "   (must be 1)")

ViT-B-16  A (5000, 157, 512)   B (1000, 109, 512)  x 4 prompts
          ||sum of components||  image 1.000000   bare 1.000000   an image of {} 1.000000   an image of a {} 1.000000   a photo of a {} 1.000000   (must be 1)
ViT-L-14  A (5000, 409, 768)   B (1000, 157, 768)  x 4 prompts


          ||sum of components||  image 1.000000   bare 1.000000   an image of {} 1.000000   an image of a {} 1.000000   a photo of a {} 1.000000   (must be 1)


## §1 — How much does each template move the accuracy, and where can that movement live?

Two facts fix the whole logic of the notebook.

**(a)** Both $z_T'(c)$ and $z_T(c)$ leave the tower already $L_2$-normalised, so the score is literally
$s(i,c)=\langle z_I(i), z_T(c)\rangle$ and
$$\Delta s(i,c) \;=\; \big\langle z_I(i),\, \Delta z_T(c)\big\rangle, \qquad
\Delta z_T(c)=\textstyle\sum_b \Delta \mathbf b_b(c).$$

**(b)** Split each component's change into a part shared by every class and a class-specific residual,
$$\Delta\mathbf b_b(c) \;=\; \underbrace{\bar{\boldsymbol\delta}_b}_{\text{mean over classes}} \;+\; \mathbf r_b(c).$$
The shared part adds the *same* number $\langle z_I,\bar{\boldsymbol\delta}\rangle$ to every class, so
$\arg\max_c$ cannot see it. **Only $\mathbf r_b(c)$ can change a prediction.**

The cell below verifies (b) as an identity, not an approximation: re-scoring with the class-mean of
$\Delta z_T$ *removed* reproduces the templated accuracy exactly, and keeping *only* the class-mean
reproduces the bare accuracy exactly — even though the shared part carries most of $\|\Delta z_T\|^2$.
The 80-template OpenAI ensemble (the classifier this repo already ships) is listed as a reference for
what prompt engineering is actually worth on these checkpoints.

In [3]:
# Goal:  §1 -- headline accuracies + the argmax-invisibility of the class-shared prompt shift.

@torch.no_grad()
def acc_of(zI, zT, lbl_v, lbl_t):
    """top-1 image->label with a raw inner product (zI is unit; zT need not be)."""
    return 100.0 * (lbl_t[(zI @ zT.T).argmax(1).cpu().numpy()] == lbl_v).mean()

rows = []
for m in MODELS:
    d = D[m]
    zI, z0 = d["A"].sum(1), d["B"]["bare"].sum(1)
    a0 = acc_of(zI, z0, d["lbl_v"], d["lbl_t"])
    # reference: the 80-template ensemble classifier the repo ships (mean of normalized per-template
    # embeddings, renormalized) -- not decomposed here, only scored.
    Z80 = torch.from_numpy(np.load(f"{OUT_DIR}/{DATASET}_classifier_{m}.npy").T.astype(np.float32)).to(device)
    Z80 = Z80 / Z80.norm(dim=1, keepdim=True)
    a80 = acc_of(zI, Z80, d["lbl_v"], np.arange(Z80.shape[0]))
    for p in VARIANTS:
        z1 = d["B"][p].sum(1)
        dz = z1 - z0
        dz_bar = dz.mean(0, keepdim=True)
        e_tot = float((dz ** 2).sum()); e_bar = float((dz_bar ** 2).sum() * dz.shape[0])
        rows.append(dict(model=m, prompt=p, acc_bare=a0, acc_prompt=acc_of(zI, z1, d["lbl_v"], d["lbl_t"]),
                         acc_80tpl_ref=a80,
                         acc_shared_only=acc_of(zI, z0 + dz_bar, d["lbl_v"], d["lbl_t"]),
                         acc_residual_only=acc_of(zI, z0 + (dz - dz_bar), d["lbl_v"], d["lbl_t"]),
                         shared_pct_of_energy=100 * e_bar / e_tot,
                         mean_norm_dz=float(dz.norm(dim=1).mean())))
        rows[-1]["gain"] = rows[-1]["acc_prompt"] - a0
df1 = pd.DataFrame(rows).set_index(["model", "prompt"]).round(3)
ACC = {(r["model"], r["prompt"]): r for r in rows}
RESULTS["acc"] = rows
print(df1[["acc_bare", "acc_prompt", "gain", "acc_80tpl_ref", "acc_shared_only",
           "acc_residual_only", "shared_pct_of_energy", "mean_norm_dz"]].to_string())
print("\nacc_shared_only == acc_bare and acc_residual_only == acc_prompt, exactly: the class-shared part")
print("of the prompt shift is argmax-invisible although it carries ~half of ||Δz_T||^2.")

                           acc_bare  acc_prompt  gain  acc_80tpl_ref  acc_shared_only  acc_residual_only  shared_pct_of_energy  mean_norm_dz
model    prompt                                                                                                                             
ViT-B-16 an image of {}       66.74       64.74 -2.00           69.5            66.74              64.74                56.757         0.496
         an image of a {}     66.74       68.02  1.28           69.5            66.74              68.02                54.666         0.423
         a photo of a {}      66.74       68.64  1.90           69.5            66.74              68.64                57.924         0.425
ViT-L-14 an image of {}       71.52       68.98 -2.54           74.6            71.52              68.98                58.193         0.574
         an image of a {}     71.52       73.46  1.94           74.6            71.52              73.46                56.582         0.478
         a ph

## §2 — Text side: is it one component that fires, or many?

Per text component $b$, three different sizes answer three different questions:

| quantity | meaning |
|---|---|
| $\mathbb E_c\|\Delta \mathbf b_b(c)\|$ | how much the component moved at all |
| $\|\bar{\boldsymbol\delta}_b\|$ | the class-**shared** part (large, and argmax-invisible) |
| $\sqrt{\mathbb E_c\|\mathbf r_b(c)\|^2}$ | the class-**varying** part — the only part that can matter |

In [4]:
# Goal:  §2 -- per text component, how big is the prompt change and how much of it is class-varying.
S2 = {}
for m in MODELS:
    d = D[m]
    for p in VARIANTS:
        dB = d["B"][p] - d["B"]["bare"]                        # [C,Kt,dim]
        dbar = dB.mean(0)                                      # [Kt,dim]  class-shared per component
        r = dB - dbar[None]
        s = dict(norm_d=dB.norm(dim=2).mean(0).cpu().numpy(),
                 norm_bar=dbar.norm(dim=1).cpu().numpy(),
                 norm_res=r.norm(dim=2).pow(2).mean(0).sqrt().cpu().numpy())
        S2[(m, p)] = s
        o_d, o_r = np.argsort(-s["norm_d"]), np.argsort(-s["norm_res"])
        print(f"=== {m}  '{p}'   Kt={len(d['nt'])} ({d['n_attn_t']} attn heads + "
              f"{len(d['nt'])-d['n_attn_t']} mlp slots)")
        print("  top-5 by ||Δb||       : " + ", ".join(f"{d['nt'][i]} {s['norm_d'][i]:.3f}" for i in o_d[:5]))
        print("  top-5 by class-varying: " + ", ".join(f"{d['nt'][i]} {s['norm_res'][i]:.3f}" for i in o_r[:5]))
        for k in (1, 5, 10):
            print(f"  top-{k:<2d} share of Σ||Δb|| = {100*s['norm_d'][o_d[:k]].sum()/s['norm_d'].sum():5.1f} %"
                  f"   |   of Σ||r|| = {100*s['norm_res'][o_r[:k]].sum()/s['norm_res'].sum():5.1f} %")
        RESULTS[f"{m}|{p}|text_top_varying"] = [(d["nt"][i], float(s["norm_res"][i])) for i in o_r[:10]]

=== ViT-B-16  'an image of {}'   Kt=109 (96 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m11 0.143, m12 0.129, m9 0.126, a8.1 0.117, m10 0.115
  top-5 by class-varying: m11 0.127, m12 0.115, m10 0.099, m9 0.098, m7 0.077
  top-1  share of Σ||Δb|| =   4.8 %   |   of Σ||r|| =   5.8 %
  top-5  share of Σ||Δb|| =  21.0 %   |   of Σ||r|| =  23.5 %
  top-10 share of Σ||Δb|| =  35.8 %   |   of Σ||r|| =  36.6 %


=== ViT-B-16  'an image of a {}'   Kt=109 (96 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m9 0.109, m12 0.104, m7 0.096, m11 0.095, a8.1 0.094
  top-5 by class-varying: m12 0.091, m9 0.086, m11 0.086, m10 0.077, m7 0.073
  top-1  share of Σ||Δb|| =   3.9 %   |   of Σ||r|| =   4.4 %
  top-5  share of Σ||Δb|| =  17.9 %   |   of Σ||r|| =  19.8 %
  top-10 share of Σ||Δb|| =  32.1 %   |   of Σ||r|| =  35.1 %


=== ViT-B-16  'a photo of a {}'   Kt=109 (96 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m9 0.103, m12 0.102, m11 0.095, m7 0.088, m6 0.084
  top-5 by class-varying: m12 0.090, m11 0.083, m9 0.082, a11.3 0.071, m10 0.071
  top-1  share of Σ||Δb|| =   4.0 %   |   of Σ||r|| =   4.5 %
  top-5  share of Σ||Δb|| =  18.2 %   |   of Σ||r|| =  19.8 %
  top-10 share of Σ||Δb|| =  31.9 %   |   of Σ||r|| =  35.5 %


=== ViT-L-14  'an image of {}'   Kt=157 (144 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m11 0.160, m12 0.159, m10 0.148, a7.9 0.121, m9 0.117
  top-5 by class-varying: m12 0.135, m11 0.131, m10 0.125, m9 0.105, m8 0.095
  top-1  share of Σ||Δb|| =   4.5 %   |   of Σ||r|| =   5.1 %
  top-5  share of Σ||Δb|| =  20.0 %   |   of Σ||r|| =  22.3 %
  top-10 share of Σ||Δb|| =  34.0 %   |   of Σ||r|| =  34.0 %


=== ViT-L-14  'an image of a {}'   Kt=157 (144 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m11 0.115, m12 0.112, m10 0.104, m9 0.097, m8 0.094
  top-5 by class-varying: m12 0.113, m10 0.097, m11 0.095, m9 0.091, m8 0.082
  top-1  share of Σ||Δb|| =   3.8 %   |   of Σ||r|| =   4.6 %
  top-5  share of Σ||Δb|| =  17.1 %   |   of Σ||r|| =  19.2 %
  top-10 share of Σ||Δb|| =  30.1 %   |   of Σ||r|| =  31.6 %


=== ViT-L-14  'a photo of a {}'   Kt=157 (144 attn heads + 13 mlp slots)
  top-5 by ||Δb||       : m12 0.111, m11 0.109, m10 0.102, m9 0.093, m8 0.089
  top-5 by class-varying: m12 0.112, m10 0.095, m11 0.093, m9 0.090, m8 0.081
  top-1  share of Σ||Δb|| =   3.8 %   |   of Σ||r|| =   4.5 %
  top-5  share of Σ||Δb|| =  17.2 %   |   of Σ||r|| =  19.1 %
  top-10 share of Σ||Δb|| =  30.0 %   |   of Σ||r|| =  31.7 %


In [5]:
# Goal:  §2 -- the picture: per-component prompt change, shared vs class-varying, over all components.
nrow = len(MODELS) * len(VARIANTS)
fig, axes = plt.subplots(nrow, 2, figsize=(15, 2.9 * nrow), squeeze=False)
for ri, (m, p) in enumerate([(m, p) for m in MODELS for p in VARIANTS]):
    d, s = D[m], S2[(m, p)]
    x = np.arange(len(d["nt"])); na = d["n_attn_t"]

    ax = axes[ri][0]
    ax.bar(x, s["norm_bar"], width=0.9, color="#c9d7ec", lw=0, label="class-shared  ‖δ̄_b‖")
    ax.plot(x, s["norm_res"], color="#b3202b", lw=1.4, label="class-varying  rms‖r_b(c)‖")
    ax.plot(x, s["norm_d"], color="#8a8a86", lw=0.8, ls=":", label="total  E_c‖Δb_b‖")
    ax.axvline(na - 0.5, color="#5a5a56", lw=0.9)
    ax.text(na + 0.6, ax.get_ylim()[1] * 0.9, "mlp slots →", fontsize=7, color="#5a5a56")
    for i in np.argsort(-s["norm_d"])[:4]:
        ax.annotate(d["nt"][i], (i, s["norm_d"][i]), fontsize=7, ha="center",
                    xytext=(0, 4), textcoords="offset points")
    ax.set_title(f"{m} — “{p}”  vs bare", fontsize=9)
    ax.set_ylabel("‖·‖ in CLIP space")
    if ri == 0: ax.legend(fontsize=7, frameon=False)
    if ri == nrow - 1: ax.set_xlabel("text component (attn heads, layer-major | mlp slots)")

    ax = axes[ri][1]
    for lbl, v, col, ls in (("total change ‖Δb‖", s["norm_d"], "#8a8a86", ":"),
                            ("class-varying part ‖r‖", s["norm_res"], "#b3202b", "-")):
        cum = np.cumsum(np.sort(v)[::-1]) / v.sum()
        ax.plot(np.arange(1, len(x) + 1), 100 * cum, color=col, lw=1.6, ls=ls, label=lbl)
    cum_r = np.cumsum(np.sort(s["norm_res"])[::-1]) / s["norm_res"].sum()
    n90 = int(np.searchsorted(cum_r, 0.9) + 1)
    ax.axhline(90, color="#c3c2b7", lw=0.8); ax.axvline(n90, color="#c3c2b7", lw=0.8)
    ax.annotate(f"90% of the class-varying change\nneeds {n90} of {len(x)} components",
                (n90, 45), fontsize=7.5, xytext=(6, 0), textcoords="offset points")
    ax.set_ylabel("cumulative share [%]"); ax.set_ylim(0, 101)
    if ri == 0: ax.legend(fontsize=7, frameon=False, loc="lower right")
    if ri == nrow - 1: ax.set_xlabel("number of text components (own descending order)")
    RESULTS[f"{m}|{p}|n_comp_for_90pct_varying"] = n90
fig.suptitle("§2  What a prompt template changes inside the text tower\n"
             "left: every component moves, the late MLP slots most · right: no small set carries it",
             fontsize=11, y=1.005)
plt.tight_layout(); save_fig(fig, "02_text_component_change"); plt.show()

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/02_text_component_change.png


## §3 — The matrices $M$, $M'$ and $\Delta M$

For a class $c$ we pair the text matrix with the **mean image component matrix of that class's own
images** ($\bar{\mathbf A}(c)$, 5 images/class) — a positive pair with the per-image noise averaged out.
Rows are text components, columns vision components; **the sum of a panel is the cosine**.

Three columns per class: $M$ (templated), $M'$ (bare), $\Delta M$; last row is the class-average
$\overline{\Delta M}$. Colour scales are **percentile-clipped at 99.5%** — a handful of MLP×MLP entries
are an order of magnitude larger than everything else and a max-based scale would render the rest blank.

In [6]:
# Goal:  §3 -- per-class M, M', dM heatmaps + the class-averaged dM, for each (model, prompt).
SHOW_CLASSES = [1, 207, 340, 417, 812, 963]   # goldfish, golden retriever, zebra, balloon, space shuttle, pizza
from utils.datasets_constants.imagenet_classes import imagenet_classes as CN

@torch.no_grad()
def class_mean_A(d):
    """[C,Kv,dim] -- the mean image component matrix of each class's own images."""
    lab = torch.as_tensor(d["lbl_v"].astype(np.int64), device=device)
    Cn = len(d["lbl_t"])
    out = torch.zeros(Cn, *d["A"].shape[1:], device=device)
    out.index_add_(0, lab, d["A"])
    cnt = torch.zeros(Cn, device=device).index_add_(0, lab, torch.ones(len(lab), device=device))
    return out / cnt.clamp_min(1)[:, None, None]

ABAR = {m: class_mean_A(D[m]) for m in MODELS}
for m in MODELS:
    d, Abar = D[m], ABAR[m]
    chk = float((torch.einsum("cbd,cad->c", d["B"]["bare"], Abar)
                 - (d["B"]["bare"].sum(1) * Abar.sum(1)).sum(1)).abs().max())
    print(f"{m}: Σ M' == cos check  {chk:.2e}")

@torch.no_grad()
def M_of(d, Abar, prompt, c=None):
    B = d["B"][prompt]
    if c is not None:
        return B[c] @ Abar[c].T                                   # [Kt,Kv] for one class
    return torch.einsum("cbd,cad->ba", B, Abar) / B.shape[0]      # class-average, never materialised

DMBAR = {}
for m in MODELS:
    d, Abar = D[m], ABAR[m]
    for p in VARIANTS:
        DMBAR[(m, p)] = M_of(d, Abar, p) - M_of(d, Abar, "bare")
        n = len(SHOW_CLASSES) + 1
        fig, axes = plt.subplots(n, 3, figsize=(13.5, 2.05 * n), squeeze=False)
        for ri in range(n):
            if ri < n - 1:
                c = SHOW_CLASSES[ri]
                m1, m0 = M_of(d, Abar, p, c), M_of(d, Abar, "bare", c)
                rlab = CN[c][:22]
            else:
                m1, m0 = M_of(d, Abar, p), M_of(d, Abar, "bare")
                rlab = "MEAN over 1000 classes"
            for cj, (Mx, ttl) in enumerate(zip([m1, m0, m1 - m0],
                                               [f"M   (“{p}”)", "M'  (bare class name)", "ΔM = M − M'"])):
                ax = axes[ri][cj]
                im, v = div_im(ax, Mx.cpu().numpy())
                cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01, extend="both")
                cb.ax.tick_params(labelsize=5.5)
                if ri == 0: ax.set_title(ttl, fontsize=8.5)
                if cj == 0: ax.set_ylabel(rlab, fontsize=7)
                ax.axhline(d["n_attn_t"] - 0.5, color="#5a5a56", lw=0.6)
                ax.axvline(d["n_attn_v"] - 0.5, color="#5a5a56", lw=0.6)
                ax.tick_params(labelsize=5.5)
                if ri == n - 1: ax.set_xlabel("vision components (attn heads | mlp)", fontsize=7)
        fig.suptitle(f"§3  {m} — “{p}”: text components (rows) × vision components (cols)\n"
                     "thin lines separate the attn block from the mlp slots; every panel sums to the cosine",
                     fontsize=11, y=1.003)
        plt.tight_layout(); save_fig(fig, f"03_M_matrices_{m}_{slug(p)}"); plt.show()

ViT-B-16: Σ M' == cos check  8.94e-08
ViT-L-14: Σ M' == cos check  8.94e-08


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-B-16_an-image-of-X.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-B-16_an-image-of-a-X.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-B-16_a-photo-of-a-X.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-L-14_an-image-of-X.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-L-14_an-image-of-a-X.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03_M_matrices_ViT-L-14_a-photo-of-a-X.png


In [7]:
# Goal:  §3b -- how concentrated is dM over pairs, and WHICH pairs carry the discriminative part.
# Does:  G_ab = E_i[ dM_ab(i, y_i) ] - E_{i,c}[ dM_ab(i, c) ]  -- the margin gain per pair.
#        sum_ab G_ab = E[ ds(i,y_i) ] - E[ mean_c ds(i,c) ], i.e. the ONLY part §1 leaves alive.

@torch.no_grad()
def gain_matrix(d, prompt, chunk=256):
    dB = d["B"][prompt] - d["B"]["bare"]
    dbar = dB.mean(0)
    lab = torch.as_tensor(d["lbl_v"].astype(np.int64), device=device)
    N = d["A"].shape[0]
    G = torch.zeros(dB.shape[1], d["A"].shape[1], dtype=torch.float64, device=device)
    Asum = torch.zeros(d["A"].shape[1], d["A"].shape[2], dtype=torch.float64, device=device)
    for i0 in range(0, N, chunk):
        sl = slice(i0, min(i0 + chunk, N))
        Ai = d["A"][sl]
        G += torch.einsum("ibd,iad->ba", dB[lab[sl]], Ai).double()
        Asum += Ai.sum(0).double()
    Gc = dbar.double() @ Asum.T                      # E_{i,c}[dM] = <dbar_b, mean_i a_a>
    return ((G - Gc) / N).float()

G3, rows = {}, []
for m in MODELS:
    d = D[m]
    for p in VARIANTS:
        G = gain_matrix(d, p); G3[(m, p)] = G
        def conc(X, ks=(1, 5, 25, 100)):
            v, _ = torch.sort(X.abs().flatten(), descending=True)
            return {f"top{k}": 100 * float(v[:k].sum()) / float(v.sum()) for k in ks}
        c_dM, c_G = conc(DMBAR[(m, p)]), conc(G)
        print(f"=== {m}  '{p}'   ({G.shape[0]}x{G.shape[1]} = {G.numel()} pairs)")
        print("  share of |ΔM̄| in top 1/5/25/100: " + ", ".join(f"{v:.1f}%" for v in c_dM.values()))
        print("  share of |G|   in top 1/5/25/100: " + ", ".join(f"{v:.1f}%" for v in c_G.values()))
        print(f"  total margin gain  Σ G_ab = {float(G.sum()):+.5f}")
        flat = G.flatten()
        for lbl, idx in (("most positive", torch.topk(flat, 8).indices),
                         ("most negative", torch.topk(-flat, 8).indices)):
            ent = [(d["nt"][int(i) // G.shape[1]], d["nv"][int(i) % G.shape[1]], float(flat[i])) for i in idx]
            print(f"  {lbl} G_ab: " + ", ".join(f"{t}×{v} {g:+.4f}" for t, v, g in ent))
            RESULTS[f"{m}|{p}|G_{lbl.replace(' ', '_')}"] = ent
        rows.append(dict(model=m, prompt=p, **{f"|ΔM̄| {k}": v for k, v in c_dM.items()},
                         **{f"|G| {k}": v for k, v in c_G.items()}, sumG=float(G.sum())))
RESULTS["concentration"] = rows
print(); print(pd.DataFrame(rows).set_index(["model", "prompt"]).round(3).to_string())

=== ViT-B-16  'an image of {}'   (109x157 = 17113 pairs)
  share of |ΔM̄| in top 1/5/25/100: 1.7%, 4.9%, 13.3%, 26.0%
  share of |G|   in top 1/5/25/100: 2.2%, 7.7%, 17.3%, 32.8%
  total margin gain  Σ G_ab = -0.00094
  most positive G_ab: a9.6×a11.5 +0.0009, a11.7×a11.5 +0.0008, a7.2×a11.5 +0.0005, a7.2×a11.4 +0.0004, a8.0×a11.7 +0.0003, m5×m12 +0.0002, a7.2×a11.7 +0.0002, m12×m11 +0.0002
  most negative G_ab: a9.4×a11.5 -0.0006, a9.5×a11.10 -0.0003, a9.1×a11.7 -0.0003, a7.2×m12 -0.0002, a9.6×m12 -0.0002, a9.5×a11.5 -0.0002, a9.4×a11.10 -0.0002, a11.7×m12 -0.0002


=== ViT-B-16  'an image of a {}'   (109x157 = 17113 pairs)
  share of |ΔM̄| in top 1/5/25/100: 1.2%, 3.9%, 12.1%, 25.4%
  share of |G|   in top 1/5/25/100: 1.4%, 5.3%, 15.0%, 31.7%
  total margin gain  Σ G_ab = +0.00068
  most positive G_ab: a9.6×a11.5 +0.0006, a8.0×a11.7 +0.0004, a7.2×a11.5 +0.0004, a11.7×a11.5 +0.0004, m12×m11 +0.0003, m5×m12 +0.0003, a7.2×a11.4 +0.0003, a5.7×a11.5 +0.0002
  most negative G_ab: a9.4×a11.5 -0.0004, a11.2×a11.10 -0.0003, a9.6×m12 -0.0002, a9.5×a11.10 -0.0002, a11.1×a11.11 -0.0002, a9.1×a11.7 -0.0002, a7.2×m12 -0.0002, a9.2×a11.6 -0.0001


=== ViT-B-16  'a photo of a {}'   (109x157 = 17113 pairs)
  share of |ΔM̄| in top 1/5/25/100: 0.8%, 3.4%, 11.7%, 26.4%
  share of |G|   in top 1/5/25/100: 1.3%, 5.0%, 14.5%, 30.7%
  total margin gain  Σ G_ab = +0.00400
  most positive G_ab: a9.6×a11.5 +0.0005, a9.0×a11.7 +0.0005, a8.0×a11.7 +0.0004, a5.7×a11.5 +0.0003, m5×m12 +0.0003, m9×m12 +0.0002, a7.2×a11.5 +0.0002, a7.7×a11.5 +0.0002
  most negative G_ab: a9.6×m12 -0.0003, m12×a11.5 -0.0002, a10.3×a11.3 -0.0002, a11.1×a11.11 -0.0002, a9.0×m12 -0.0002, a10.3×a11.7 -0.0001, a5.7×m12 -0.0001, a8.0×m12 -0.0001


=== ViT-L-14  'an image of {}'   (157x409 = 64213 pairs)
  share of |ΔM̄| in top 1/5/25/100: 1.1%, 3.2%, 7.9%, 16.7%
  share of |G|   in top 1/5/25/100: 1.0%, 2.9%, 8.6%, 19.2%
  total margin gain  Σ G_ab = -0.00792
  most positive G_ab: a9.11×a23.2 +0.0006, a10.4×a22.6 +0.0003, a9.11×a22.6 +0.0003, a10.9×a23.2 +0.0002, a10.4×a23.9 +0.0002, a10.4×a23.2 +0.0002, a10.6×a23.7 +0.0002, a9.11×a22.0 +0.0002
  most negative G_ab: a10.0×a23.14 -0.0003, a8.7×a23.8 -0.0003, a11.0×a23.7 -0.0002, a11.5×a23.2 -0.0002, a11.0×a23.1 -0.0002, a10.0×a22.6 -0.0002, a9.11×m24 -0.0002, a10.1×a23.0 -0.0001


=== ViT-L-14  'an image of a {}'   (157x409 = 64213 pairs)
  share of |ΔM̄| in top 1/5/25/100: 0.7%, 2.4%, 6.7%, 14.6%
  share of |G|   in top 1/5/25/100: 0.8%, 2.8%, 8.4%, 17.9%
  total margin gain  Σ G_ab = -0.00445
  most positive G_ab: m12×m24 +0.0004, a10.4×a22.6 +0.0003, m12×a23.1 +0.0003, m12×m23 +0.0002, a10.4×a23.2 +0.0002, m11×m24 +0.0002, a10.4×a23.9 +0.0002, m12×a23.14 +0.0002
  most negative G_ab: a10.4×m24 -0.0002, a11.7×a23.15 -0.0001, a11.5×a23.2 -0.0001, a8.7×a23.8 -0.0001, a4.6×a23.1 -0.0001, a11.0×a23.7 -0.0001, a10.10×a23.6 -0.0001, a9.10×a23.8 -0.0001


=== ViT-L-14  'a photo of a {}'   (157x409 = 64213 pairs)
  share of |ΔM̄| in top 1/5/25/100: 0.8%, 2.6%, 7.1%, 15.0%
  share of |G|   in top 1/5/25/100: 0.6%, 2.4%, 7.9%, 17.2%
  total margin gain  Σ G_ab = +0.00255
  most positive G_ab: a10.6×a23.7 +0.0003, a10.4×a22.6 +0.0003, m12×m24 +0.0002, a10.4×a23.9 +0.0002, a5.2×a23.1 +0.0002, a10.4×a23.2 +0.0002, a9.11×a23.2 +0.0002, a10.9×a23.2 +0.0002
  most negative G_ab: a10.4×m24 -0.0002, a11.7×a23.15 -0.0002, a11.5×a23.2 -0.0002, a6.5×m24 -0.0002, a10.6×m24 -0.0001, a9.11×m24 -0.0001, a4.6×a23.1 -0.0001, a5.2×m24 -0.0001

                           |ΔM̄| top1  |ΔM̄| top5  |ΔM̄| top25  |ΔM̄| top100  |G| top1  |G| top5  |G| top25  |G| top100   sumG
model    prompt                                                                                                               
ViT-B-16 an image of {}         1.691       4.864       13.300        26.006     2.181     7.698     17.262      32.770 -0.001
         an image of a {}       1.165   

In [8]:
# Goal:  §3c -- the raw class-averaged ΔM̄ next to G, the slice of it argmax can see, with marginals.
keys = [(m, p) for m in MODELS for p in VARIANTS]
fig, axes = plt.subplots(2, 2 * len(keys), figsize=(3.4 * 2 * len(keys), 6.4), squeeze=False)
for ci, (m, p) in enumerate(keys):
    d = D[m]
    for k, (X, ttl) in enumerate([(DMBAR[(m, p)], "ΔM̄  class-averaged"), (G3[(m, p)], "G  discriminative")]):
        Xn = X.cpu().numpy()
        ax = axes[0][2 * ci + k]
        im, _ = div_im(ax, Xn)
        ax.set_title(f"{m}\n“{p}”\n{ttl}", fontsize=7.5)
        ax.axhline(d["n_attn_t"] - 0.5, color="#5a5a56", lw=0.6)
        ax.axvline(d["n_attn_v"] - 0.5, color="#5a5a56", lw=0.6)
        ax.tick_params(labelsize=6)
        if 2 * ci + k == 0: ax.set_ylabel("text components", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01, extend="both").ax.tick_params(labelsize=5.5)

        ax = axes[1][2 * ci + k]
        rowm, colm = Xn.sum(1), Xn.sum(0)
        ax.plot(np.arange(len(colm)), colm, color="#2a78d6", lw=1.0)
        ax2 = ax.twiny(); ax2.plot(np.arange(len(rowm)), rowm, color="#b3202b", lw=1.0)
        ax2.tick_params(labelsize=6, colors="#b3202b"); ax2.grid(False)
        ax.tick_params(labelsize=6); ax.axhline(0, color="#8a8a86", lw=0.8)
        if 2 * ci + k == 0:
            ax.set_ylabel("marginal sum", fontsize=7)
            ax.set_xlabel("vision comp.", fontsize=7, color="#2a78d6")
            ax2.set_xlabel("text comp.", fontsize=7, color="#b3202b")
        for i in np.argsort(-np.abs(rowm))[:3]:
            ax2.annotate(d["nt"][i], (i, rowm[i]), fontsize=6.5, color="#b3202b",
                         xytext=(0, 3), textcoords="offset points", ha="center")
        for i in np.argsort(-np.abs(colm))[:3]:
            ax.annotate(d["nv"][i], (i, colm[i]), fontsize=6.5, color="#2a78d6",
                        xytext=(0, -9), textcoords="offset points", ha="center")
fig.suptitle("§3c  The prompt change over component pairs, and the slice of it that argmax can see\n"
             "top: pair matrices (blue negative / orange positive, 99.5-percentile scale) · "
             "bottom: marginals, blue = vision axis (bottom ticks), red = text axis (top ticks)",
             fontsize=11, y=1.03)
plt.tight_layout(); save_fig(fig, "03c_dM_vs_G"); plt.show()

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/03c_dM_vs_G.png


## §4 — The (cos, weight) plane, one series per prompt

The §8 figure, with the series being the **prompts** instead of the polarities; the polarity becomes the
panel, so positive and negative pairs are both shown.

* **Vision heads vs the text embedding** — the vision tower is *unchanged*, so the weight
  $\|\hat a\|/\|z_I\|$ is identical in every series by construction and the points can only slide
  **horizontally**. That horizontal slide is exactly the "change in direction" the question asks about.
* **Text components vs the image embedding** — here both coordinates move, because the prompt rewrites
  the text components themselves.

$\varphi=\text{weight}\times\cos$ is the component's additive share of the cosine; dotted hyperbolae are
contours of constant $\varphi$. The negative partner is drawn **once** per (model, polarity) and reused
across prompts, so a point's move is the prompt and nothing else.

In [9]:
# Goal:  §4 -- (cos, weight) per component for every prompt, positive and negative pairs.
def pair_classes(d, polarity, Cn):
    lab = d["lbl_v"]
    if polarity == "pos":
        return lab.copy()
    cc = np.random.default_rng(0).integers(0, Cn, size=len(lab))
    bad = cc == lab
    cc[bad] = (cc[bad] + 1) % Cn
    return cc

@torch.no_grad()
def plane(d, side, prompt, polarity, chunk=512):
    """(weight, cos) per component of `side` against the OTHER tower's full embedding."""
    B = d["B"][prompt]
    cc = torch.as_tensor(pair_classes(d, polarity, B.shape[0]).astype(np.int64), device=device)
    zI, zT = d["A"].sum(1), B.sum(1)
    N = len(cc)
    K = d["A"].shape[1] if side == "vision" else B.shape[1]
    w_acc = torch.zeros(K, dtype=torch.float64, device=device)
    c_acc = torch.zeros(K, dtype=torch.float64, device=device)
    for i0 in range(0, N, chunk):
        sl = slice(i0, min(i0 + chunk, N))
        if side == "vision":
            X, zself, zoth = d["A"][sl], zI[sl], zT[cc[sl]]
        else:
            X, zself, zoth = B[cc[sl]], zT[cc[sl]], zI[sl]
        nX = X.norm(dim=-1)
        w_acc += (nX / zself.norm(dim=-1)[:, None]).double().sum(0)
        c_acc += (torch.einsum("iad,id->ia", X, zoth / zoth.norm(dim=-1, keepdim=True))
                  / (nX + 1e-30)).double().sum(0)
    return (w_acc / N).cpu().numpy(), (c_acc / N).cpu().numpy()

PL = {(m, s, p, q): plane(D[m], s, p, q)
      for m in MODELS for s in ("vision", "text") for p in PROMPTS for q in ("pos", "neg")}

def panel(ax, m, side, prompt, polarity):
    """One panel = bare vs ONE prompt, so every panel carries exactly two (validated) series."""
    d = D[m]
    K = d["n_attn_v"] if side == "vision" else d["n_attn_t"]      # attn heads only, as in §8
    sl = slice(0, K)
    w0, c0 = PL[(m, side, "bare", polarity)]
    w1, c1 = PL[(m, side, prompt, polarity)]
    for lab, (w, c), col in (("bare", (w0, c0), P_COLOR["bare"]), (prompt, (w1, c1), P_COLOR[prompt])):
        ax.scatter(c[sl], w[sl], s=11, c=col, marker=P_MARKER[lab], alpha=0.75, lw=0, label=lab)
    ax.plot([c0[sl], c1[sl]], [w0[sl], w1[sl]], color="#c3c2b7", lw=0.4, zorder=0)
    ax.axvline(0, color="#8a8a86", lw=0.8, zorder=0)
    lo, hi = ax.get_xlim(); wmax = float(max(w0[sl].max(), w1[sl].max())) * 1.15
    ax.set_xlim(lo, hi); ax.set_ylim(0, wmax)
    phimax = float((w0[sl] * c0[sl]).max())
    for phi in (0.25 * phimax, 0.5 * phimax, phimax):
        xs = np.linspace(max(lo, 1e-4), hi, 400); ys = phi / np.clip(xs, 1e-9, None)
        ok = (xs > 0) & (ys <= wmax)
        if ok.any(): ax.plot(xs[ok], ys[ok], color="#c3c2b7", lw=0.6, ls=":", zorder=0)
    names = d["nv"] if side == "vision" else d["nt"]
    for i in np.argsort(-np.abs(w1[sl] * c1[sl] - w0[sl] * c0[sl]))[:4]:
        ax.annotate(names[i], (c1[i], w1[i]), fontsize=6.5, xytext=(3, 3), textcoords="offset points")
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=6, loc="upper left", frameon=False)

cols = [(m, pol) for m in MODELS for pol in ("pos", "neg")]
for side, other in (("vision", "text"), ("text", "image")):
    fig, axes = plt.subplots(len(VARIANTS), len(cols), figsize=(4.0 * len(cols), 3.2 * len(VARIANTS)),
                             squeeze=False)
    for ri, p in enumerate(VARIANTS):
        for ci, (m, pol) in enumerate(cols):
            ax = axes[ri][ci]; panel(ax, m, side, p, pol)
            if ri == 0: ax.set_title(f"{m} — {'positive' if pol == 'pos' else 'negative'} pairs", fontsize=8.5)
            if ci == 0: ax.set_ylabel(f"“{p}”\nweight  ‖â‖/‖ẑ‖", fontsize=7.5)
            if ri == len(VARIANTS) - 1:
                ax.set_xlabel(f"cos_sim  cos(â, ẑ_{'T' if side == 'vision' else 'I'})", fontsize=7)
    fig.suptitle(f"§4  {side} attention heads vs the {other} tower's full embedding — φ = weight × cos_sim\n"
                 "each head joined from its bare position to its position under the prompt",
                 fontsize=11, y=1.005)
    plt.tight_layout(); save_fig(fig, f"04_plane_{side}"); plt.show()

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/04_plane_vision.png


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/04_plane_text.png


In [10]:
# Goal:  §4b -- do the top interactions stay the same, and who moves in DIRECTION vs in MAGNITUDE?
rows = []
for m in MODELS:
    d = D[m]
    for side in ("vision", "text"):
        K = d["n_attn_v"] if side == "vision" else d["n_attn_t"]
        names = d["nv"] if side == "vision" else d["nt"]
        w0, c0 = PL[(m, side, "bare", "pos")]
        for p in VARIANTS:
            for pol in ("pos", "neg"):
                a0 = PL[(m, side, "bare", pol)]; a1 = PL[(m, side, p, pol)]
                p_0, p_1 = a0[0] * a0[1], a1[0] * a1[1]
                rows.append(dict(model=m, side=side, prompt=p, polarity=pol,
                                 spearman_phi=spearmanr(p_0[:K], p_1[:K]).statistic,
                                 top15_overlap=len(set(np.argsort(-p_0[:K])[:15]) &
                                                   set(np.argsort(-p_1[:K])[:15])),
                                 mean_dcos=float(np.mean(a1[1][:K] - a0[1][:K])),
                                 mean_abs_dcos=float(np.mean(np.abs(a1[1][:K] - a0[1][:K]))),
                                 mean_abs_dweight=float(np.mean(np.abs(a1[0][:K] - a0[0][:K]))),
                                 sum_dphi=float(np.sum(p_1 - p_0))))
            a1 = PL[(m, side, p, "pos")]
            mv = np.argsort(-np.abs(a1[1][:K] - c0[:K]))[:6]
            print(f"{m:9s} {side:6s} '{p:16s}' biggest Δcos (pos): " +
                  ", ".join(f"{names[i]} {a1[1][i]-c0[i]:+.3f}" for i in mv))
df4 = pd.DataFrame(rows).round(4)
print(); print(df4.to_string(index=False))
RESULTS["plane_summary"] = df4.to_dict("records")

ViT-B-16  vision 'an image of {}  ' biggest Δcos (pos): a5.8 -0.033, a8.7 -0.030, a1.10 +0.029, a8.10 +0.029, a3.4 -0.028, a5.1 +0.028
ViT-B-16  vision 'an image of a {}' biggest Δcos (pos): a9.5 -0.028, a7.3 +0.028, a7.0 +0.027, a5.1 +0.026, a8.10 +0.025, a9.0 -0.024
ViT-B-16  vision 'a photo of a {} ' biggest Δcos (pos): a9.9 -0.052, a5.0 +0.039, a9.5 -0.039, a7.3 +0.036, a9.0 -0.036, a6.5 +0.035
ViT-B-16  text   'an image of {}  ' biggest Δcos (pos): a8.5 +0.072, a10.5 +0.065, a8.6 +0.038, a9.3 +0.036, a2.5 +0.034, a1.1 +0.030
ViT-B-16  text   'an image of a {}' biggest Δcos (pos): a8.5 +0.065, a8.3 +0.058, a10.5 +0.057, a5.3 +0.055, a1.1 +0.047, a6.7 -0.033
ViT-B-16  text   'a photo of a {} ' biggest Δcos (pos): a8.5 +0.107, a8.1 +0.076, a8.3 +0.075, a6.7 +0.055, a5.3 +0.051, a10.5 +0.049
ViT-L-14  vision 'an image of {}  ' biggest Δcos (pos): a19.10 -0.041, a20.12 -0.038, a3.14 +0.038, a17.6 -0.037, a3.2 +0.036, a19.15 -0.036
ViT-L-14  vision 'an image of a {}' biggest Δcos (pos):

## §5 — The causal test: which components actually move the accuracy?

Build a **hybrid** text embedding — bare, with the templated version of a subset $S$ of text components
swapped in,
$$z_T^{S}(c) \;=\; z_T'(c) \;+\; \sum_{b\in S} \Delta \mathbf b_b(c),$$
exactly bare at $S=\varnothing$ and exactly templated at $S=$ all. Scored with the raw inner product, so
the interpolation is additive and matches the $\Delta M$ bookkeeping; the re-normalised variant is drawn
alongside as a check.

Three read-outs: **(a)** one component at a time, **(b)** cumulative swap-in under several orderings,
**(c)** the vision side — restrict $\Delta s$ to the top vision components and see how much of the effect
still gets through.

In [11]:
# Goal:  §5a -- swap in ONE text component at a time; Δ accuracy against bare.
@torch.no_grad()
def acc_from_zT(zI, zT, lbl_v, lbl_t, renorm=False):
    z = zT / zT.norm(dim=1, keepdim=True) if renorm else zT
    return 100.0 * (lbl_t[(zI @ z.T).argmax(1).cpu().numpy()] == lbl_v).mean()

S5 = {}
for m in MODELS:
    d = D[m]
    zI, z0 = d["A"].sum(1), d["B"]["bare"].sum(1)
    base = acc_from_zT(zI, z0, d["lbl_v"], d["lbl_t"])
    for p in VARIANTS:
        dB = d["B"][p] - d["B"]["bare"]
        single = np.array([acc_from_zT(zI, z0 + dB[:, b], d["lbl_v"], d["lbl_t"]) - base
                           for b in range(dB.shape[1])])
        S5[(m, p)] = dict(base=base, single=single)
        o = np.argsort(-single); g = ACC[(m, p)]["gain"]
        print(f"=== {m}  '{p}'   bare {base:.2f}% -> {ACC[(m,p)]['acc_prompt']:.2f}%  (gain {g:+.2f})")
        print("  best  single swaps: " + ", ".join(f"{d['nt'][i]} {single[i]:+.2f}" for i in o[:6]))
        print("  worst single swaps: " + ", ".join(f"{d['nt'][i]} {single[i]:+.2f}" for i in o[-5:]))
        print(f"  Σ single-swap effects = {single.sum():+.2f}  vs the real joint effect {g:+.2f}")
        RESULTS[f"{m}|{p}|single_swap"] = [(d["nt"][i], float(single[i])) for i in o[:10]] + \
                                          [(d["nt"][i], float(single[i])) for i in o[-5:]]

=== ViT-B-16  'an image of {}'   bare 66.74% -> 64.74%  (gain -2.00)
  best  single swaps: a11.1 +0.32, m9 +0.26, a6.6 +0.24, a7.5 +0.20, m8 +0.16, a10.2 +0.16
  worst single swaps: a11.7 -0.56, m6 -0.56, a9.6 -0.62, m11 -1.38, m12 -4.24
  Σ single-swap effects = -6.30  vs the real joint effect -2.00


=== ViT-B-16  'an image of a {}'   bare 66.74% -> 68.02%  (gain +1.28)
  best  single swaps: m9 +0.50, m7 +0.38, a5.7 +0.24, a7.5 +0.22, a11.0 +0.20, a10.2 +0.18
  worst single swaps: a8.5 -0.26, a11.7 -0.28, m11 -0.30, m6 -0.56, m12 -3.10
  Σ single-swap effects = -0.68  vs the real joint effect +1.28


=== ViT-B-16  'a photo of a {}'   bare 66.74% -> 68.64%  (gain +1.90)
  best  single swaps: m9 +0.80, m7 +0.44, a9.5 +0.34, a7.5 +0.28, a7.3 +0.24, a6.6 +0.22
  worst single swaps: m4 -0.26, a10.3 -0.28, m6 -0.34, a11.7 -0.42, m12 -3.06
  Σ single-swap effects = +1.60  vs the real joint effect +1.90


=== ViT-L-14  'an image of {}'   bare 71.52% -> 68.98%  (gain -2.54)
  best  single swaps: m8 +0.28, a5.11 +0.24, a8.11 +0.20, a5.2 +0.16, a10.2 +0.14, a7.4 +0.14
  worst single swaps: m7 -0.52, m3 -0.56, m5 -0.58, a10.0 -0.80, m12 -2.98
  Σ single-swap effects = -9.46  vs the real joint effect -2.54


=== ViT-L-14  'an image of a {}'   bare 71.52% -> 73.46%  (gain +1.94)
  best  single swaps: a4.4 +0.20, a5.5 +0.16, a7.9 +0.16, a5.6 +0.16, a4.7 +0.14, a7.3 +0.14
  worst single swaps: a10.4 -0.44, a11.4 -0.56, m3 -0.60, a10.0 -0.64, m12 -2.16
  Σ single-swap effects = -5.82  vs the real joint effect +1.94


=== ViT-L-14  'a photo of a {}'   bare 71.52% -> 74.00%  (gain +2.48)
  best  single swaps: m11 +0.28, m10 +0.22, m6 +0.18, a1.3 +0.16, a4.4 +0.16, a4.7 +0.16
  worst single swaps: a10.4 -0.52, m3 -0.58, a10.0 -0.60, a11.4 -0.72, m12 -2.52
  Σ single-swap effects = -7.06  vs the real joint effect +2.48


In [12]:
# Goal:  §5b -- cumulative swap-in under 5 orderings + the single-swap bar chart.
@torch.no_grad()
def sweep(d, p, order, n_ckpt=24):
    zI, z0 = d["A"].sum(1), d["B"]["bare"].sum(1)
    dB = d["B"][p] - d["B"]["bare"]
    Kt = dB.shape[1]
    ks = np.unique(np.r_[0, np.round(np.linspace(0, Kt, n_ckpt)).astype(int)])
    acc, accn, z, prev = [], [], z0.clone(), 0
    for k in ks:
        if k > prev:
            sel = torch.as_tensor(np.ascontiguousarray(order[prev:k]).astype(np.int64), device=device)
            z = z + dB[:, sel].sum(1); prev = k
        acc.append(acc_from_zT(zI, z, d["lbl_v"], d["lbl_t"]))
        accn.append(acc_from_zT(zI, z, d["lbl_v"], d["lbl_t"], renorm=True))
    return ks, np.array(acc), np.array(accn)

ORD_COLOR = {"by single-swap Δacc": "#2a78d6", "by class-varying ‖r_b‖": "#b3202b",
             "by total ‖Δb_b‖": "#eb6834", "reverse of Δacc": "#1baf7a"}
keys = [(m, p) for m in MODELS for p in VARIANTS]
# rows alternate per model: sweep row then single-swap row, so panels stay wide enough to read
fig, axes = plt.subplots(2 * len(MODELS), len(VARIANTS),
                         figsize=(5.6 * len(VARIANTS), 3.3 * 2 * len(MODELS)), squeeze=False)
for ci, (m, p) in enumerate(keys):
    r0, cc = 2 * (ci // len(VARIANTS)), ci % len(VARIANTS)
    d, s2, s5 = D[m], S2[(m, p)], S5[(m, p)]
    Kt = len(d["nt"])
    orders = {"by single-swap Δacc": np.argsort(-s5["single"]),
              "by class-varying ‖r_b‖": np.argsort(-s2["norm_res"]),
              "by total ‖Δb_b‖": np.argsort(-s2["norm_d"]),
              "reverse of Δacc": np.argsort(s5["single"])}
    ax = axes[r0][cc]
    for lbl, o in orders.items():
        ks, a, an = sweep(d, p, o)
        ax.plot(ks, a, color=ORD_COLOR[lbl], lw=1.6, label=lbl)
        if lbl == "by single-swap Δacc":
            ax.plot(ks, an, color=ORD_COLOR[lbl], lw=1.0, ls="--", label="  ↳ same, re-normalised")
    R = np.stack([sweep(d, p, np.random.default_rng(s).permutation(Kt))[1] for s in range(5)])
    ax.plot(ks, R.mean(0), color="#8a8a86", lw=1.2, ls=":", label="random (5 runs)")
    ax.fill_between(ks, R.min(0), R.max(0), color="#8a8a86", alpha=0.15, lw=0)
    for y, t in ((s5["base"], "bare"), (ACC[(m, p)]["acc_prompt"], "templated")):
        ax.axhline(y, color="#c3c2b7", lw=0.9)
        ax.annotate(t, (Kt * 0.72, y), fontsize=7, xytext=(0, 3), textcoords="offset points")
    ax.set_title(f"{m} — bare → “{p}”", fontsize=9)
    ax.set_xlabel("number of text components swapped in"); ax.set_ylabel("zero-shot top-1 [%]")
    if ci == 0: ax.legend(fontsize=6.5, frameon=False, loc="lower right")

    ax = axes[r0 + 1][cc]
    ax.bar(np.arange(Kt), s5["single"], width=0.86, lw=0, color=[F_COLOR[k] for k in d["kt"][0]])
    ax.axhline(0, color="#8a8a86", lw=0.8)
    ax.axvline(d["n_attn_t"] - 0.5, color="#5a5a56", lw=0.9)
    for i in np.argsort(-np.abs(s5["single"]))[:5]:
        ax.annotate(d["nt"][i], (i, s5["single"][i]), fontsize=7, ha="center",
                    xytext=(0, 4 if s5["single"][i] > 0 else -10), textcoords="offset points")
    ax.set_xlabel("text component"); ax.set_ylabel("Δ top-1 vs bare [%]")
    ax.set_title("swapping exactly ONE component", fontsize=9)
    if ci == 0:
        ax.legend(handles=[Line2D([], [], color=F_COLOR[k], lw=6, label=k) for k in ("attn", "mlp")],
                  fontsize=7, frameon=False, loc="lower left")
fig.suptitle("§5  Which text components actually move the zero-shot accuracy\n"
             "top: cumulative swap-in · bottom: the effect of each component on its own",
             fontsize=11, y=1.01)
plt.tight_layout(); save_fig(fig, "05_swap_in"); plt.show()

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/05_swap_in.png


In [13]:
# Goal:  §5c -- the VISION side: how much of the effect flows through how many vision components?
# Does:  score = <z_I, z_T'> + sum_{a in topK} <a, dz_T>.  The vision tower never changed, so this asks
#        which vision components RECEIVE the rewritten text signal.
# Read the curves, not an interpolation: the partial sums are NOT between the two endpoints. The
# per-component terms <a, dz_T> are individually far larger than their sum, so truncating the sum
# leaves a large uncancelled bias and the accuracy dives below both ends. The cancellation ratio
# printed below quantifies exactly that, and is the number to quote instead of a "K for 80%".
rows = []
fig, axes = plt.subplots(len(MODELS), len(VARIANTS), figsize=(5.0 * len(VARIANTS), 3.4 * len(MODELS)),
                         squeeze=False)
for ci, (m, p) in enumerate(keys):
    d = D[m]
    ax = axes[ci // len(VARIANTS)][ci % len(VARIANTS)]
    zI, z0 = d["A"].sum(1), d["B"]["bare"].sum(1)
    dz = (d["B"][p] - d["B"]["bare"]).sum(1)
    base, full = S5[(m, p)]["base"], ACC[(m, p)]["acc_prompt"]
    gcol = G3[(m, p)].sum(0).cpu().numpy()
    Kv = len(gcol)
    ks = np.unique(np.r_[0, np.round(np.geomspace(1, Kv, 18)).astype(int)])
    for lbl, order, col in (("top |G| vision comps", np.argsort(-np.abs(gcol)), "#2a78d6"),
                            ("largest ‖a‖ first", np.argsort(-d["A"].norm(dim=2).mean(0).cpu().numpy()), "#eb6834"),
                            ("random", np.random.default_rng(0).permutation(Kv), "#8a8a86")):
        accs = []
        for k in ks:
            sel = torch.as_tensor(np.ascontiguousarray(order[:k]).astype(np.int64), device=device)
            part = d["A"][:, sel].sum(1) @ dz.T if k else 0.0
            accs.append(acc_from_zT(zI, z0, d["lbl_v"], d["lbl_t"]) if k == 0 else
                        100.0 * (d["lbl_t"][(zI @ z0.T + part).argmax(1).cpu().numpy()] == d["lbl_v"]).mean())
        ax.plot(ks, accs, color=col, lw=1.6, ls=":" if lbl == "random" else "-", label=lbl)
    ax.axhline(base, color="#c3c2b7", lw=0.9); ax.axhline(full, color="#c3c2b7", lw=0.9)
    ax.set_xscale("log"); ax.set_title(f"{m} — “{p}”", fontsize=9)
    if ci >= len(keys) - len(VARIANTS): ax.set_xlabel("vision comps allowed to receive Δz_T (log)")
    if ci % len(VARIANTS) == 0: ax.set_ylabel("zero-shot top-1 [%]")
    if ci == 0: ax.legend(fontsize=7, frameon=False, loc="lower right")
    # how much the per-component terms cancel: Σ|term| / |Σ term|. 1 = no cancellation.
    canc = float(np.abs(gcol).sum() / (abs(gcol.sum()) + 1e-30))
    rows.append(dict(model=m, prompt=p, Kv=Kv, cancellation_ratio=canc,
                     top10_share_of_abs=100 * float(np.sort(np.abs(gcol))[::-1][:10].sum() / np.abs(gcol).sum())))
    top = np.argsort(-np.abs(gcol))[:8]
    print(f"{m:9s} '{p:16s}' top vision receivers: " + ", ".join(f"{d['nv'][i]} {gcol[i]:+.4f}" for i in top))
    RESULTS[f"{m}|{p}|vision_receivers"] = [(d["nv"][i], float(gcol[i])) for i in top]
fig.suptitle("§5c  Letting only K vision components receive Δz_T — the partial sums are NOT between the\n"
             "two endpoints, because the per-component terms are large and mutually cancelling\n"
             "(grey lines = bare and fully-templated accuracy)", fontsize=10.5, y=1.04)
plt.tight_layout(); save_fig(fig, "05c_vision_receivers"); plt.show()
print("\ncancellation_ratio = Σ_a|G_a| / |Σ_a G_a| over vision components (1 = no cancellation)")
print(pd.DataFrame(rows).round(2).to_string(index=False))
RESULTS["vision_cancellation"] = rows

ViT-B-16  'an image of {}  ' top vision receivers: a11.5 +0.0012, a11.10 -0.0010, a11.4 +0.0007, a11.8 -0.0004, a11.3 -0.0003, a11.2 -0.0002, a11.1 -0.0002, m12 +0.0002


ViT-B-16  'an image of a {}' top vision receivers: a11.10 -0.0010, a11.5 +0.0008, m12 +0.0008, a11.4 +0.0007, a10.10 +0.0003, a11.8 -0.0003, a11.1 -0.0003, a11.11 -0.0003


ViT-B-16  'a photo of a {} ' top vision receivers: a11.4 +0.0005, m11 +0.0005, a11.5 +0.0005, a11.7 +0.0004, a10.10 +0.0004, a11.1 -0.0003, a10.5 +0.0003, m12 +0.0002


ViT-L-14  'an image of {}  ' top vision receivers: m24 +0.0012, a23.1 -0.0009, a23.7 -0.0008, a23.14 -0.0008, a23.8 -0.0008, a22.3 -0.0003, a22.10 -0.0003, a22.4 -0.0003


ViT-L-14  'an image of a {}' top vision receivers: m24 +0.0011, a23.1 -0.0006, a23.8 -0.0006, a23.2 -0.0004, a23.7 -0.0004, a23.15 -0.0003, a23.14 -0.0003, a22.10 -0.0003


ViT-L-14  'a photo of a {} ' top vision receivers: a23.15 -0.0004, a23.7 +0.0003, m23 +0.0003, a17.5 +0.0002, a22.6 +0.0002, a23.0 +0.0002, m22 +0.0001, a22.11 +0.0001


saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/05c_vision_receivers.png

cancellation_ratio = Σ_a|G_a| / |Σ_a G_a| over vision components (1 = no cancellation)
   model           prompt  Kv  cancellation_ratio  top10_share_of_abs
ViT-B-16   an image of {} 157                7.54               63.00
ViT-B-16 an image of a {} 157               11.48               63.56
ViT-B-16  a photo of a {} 157                1.70               51.19
ViT-L-14   an image of {} 409                1.60               47.24
ViT-L-14 an image of a {} 409                2.35               42.36
ViT-L-14  a photo of a {} 409                2.73               29.34


## §6 — Does the template move the text embeddings *towards* the images?

The last question. Measured five ways — the mean cosine over all (image, class) pairs (the modality gap),
the mean cosine on positive pairs, the **margin** against the class average and against the best wrong
class (the quantity argmax actually ranks on), and the geometry of the centroids. The point of the table
is that the first two can move a lot while the accuracy follows only the last.

In [14]:
# Goal:  §6 -- modality-gap and margin statistics for every prompt.
rows = []
for m in MODELS:
    d = D[m]
    zI = d["A"].sum(1)
    lab = torch.as_tensor(d["lbl_v"].astype(np.int64), device=device)
    for p in PROMPTS:
        zT = d["B"][p].sum(1)
        sim = zI @ zT.T                                            # both unit -> cosine
        pos = sim[torch.arange(len(lab), device=device), lab]
        allm = sim.mean()
        best_wrong = sim.scatter(1, lab[:, None], -1e9).max(1).values
        tt = zT @ zT.T
        off = (tt.sum() - tt.diagonal().sum()) / (tt.shape[0] * (tt.shape[0] - 1))
        cI, cT = zI.mean(0), zT.mean(0)
        rows.append(dict(model=m, prompt=p, cos_all=float(allm), cos_pos=float(pos.mean()),
                         margin_vs_mean=float(pos.mean() - allm),
                         margin_vs_best_wrong=float((pos - best_wrong).mean()),
                         cos_text_text=float(off), centroid_gap=float((cI - cT).norm()),
                         acc=(ACC[(m, p)]["acc_prompt"] if p in VARIANTS else ACC[(m, PROMPTS[1])]["acc_bare"])))
df6 = pd.DataFrame(rows).set_index(["model", "prompt"]).round(4)
print(df6.to_string())
RESULTS["geometry"] = df6.reset_index().to_dict("records")

metrics = [("cos_all", "mean cos over ALL pairs\n(the modality gap)"),
           ("cos_pos", "mean cos on POSITIVE pairs"),
           ("margin_vs_best_wrong", "margin vs the best WRONG class\n(what decides the argmax)"),
           ("acc", "zero-shot top-1 [%]")]
fig, axes = plt.subplots(1, len(metrics), figsize=(3.6 * len(metrics), 3.3))
xs = np.arange(len(MODELS)); wdt = 0.26
for ax, (k, ttl) in zip(axes, metrics):
    for j, p in enumerate(PROMPTS):
        v = [df6.loc[(m, p), k] for m in MODELS]
        b = ax.bar(xs + (j - 1) * wdt, v, wdt * 0.82, color=P_COLOR[p], lw=0, label=p)  # 12% surface gap
        ax.bar_label(b, fmt="%.3f" if k != "acc" else "%.1f", fontsize=6, padding=2)
    ax.set_xticks(xs); ax.set_xticklabels(MODELS, fontsize=7.5)
    ax.set_title(ttl, fontsize=8.5); ax.grid(axis="x", alpha=0)
axes[0].legend(fontsize=7, frameon=False, title="text prompt", title_fontsize=7)
fig.suptitle("§6  Where the prompt moves the text embeddings, and what the accuracy follows",
             fontsize=10.5, y=1.04)
plt.tight_layout(); save_fig(fig, "06_geometry"); plt.show()

with open(f"{FIG_DIR}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=1, default=float)
print("\nwrote", f"{FIG_DIR}/results.json")

                           cos_all  cos_pos  margin_vs_mean  margin_vs_best_wrong  cos_text_text  centroid_gap    acc
model    prompt                                                                                                      
ViT-B-16 bare               0.0900   0.2689          0.1789                0.0164         0.5076        0.8090  66.74
         an image of {}     0.0697   0.2477          0.1780                0.0138         0.5076        0.8337  64.74
         an image of a {}   0.0794   0.2590          0.1796                0.0166         0.5806        0.8653  68.02
         a photo of a {}    0.0868   0.2698          0.1829                0.0179         0.5783        0.8552  68.64
ViT-L-14 bare               0.0986   0.2860          0.1873                0.0235         0.5090        0.8317  71.52
         an image of {}     0.0828   0.2623          0.1794                0.0212         0.5347        0.8654  68.98
         an image of a {}   0.0938   0.2767          0.1

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/06_geometry.png

wrote output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/results.json


## §7 — Reading

### The premise had to be fixed first

`"an image of {}"` does not raise ImageNet zero-shot on these LAION checkpoints — it **lowers** it,
by 2.00 (ViT-B-16) and 2.54 (ViT-L-14) points against the bare class name. What raises it is the
indefinite article:

| prompt | ViT-B-16 | ViT-L-14 |
|---|---|---|
| bare class name | 66.74 | 71.52 |
| `an image of {}` | 64.74 (−2.00) | 68.98 (−2.54) |
| `a photo of {}` | 67.18 (+0.44) | 72.52 (+1.00) |
| `an image of a {}` | 68.02 (+1.28) | 73.46 (+1.94) |
| `a photo of a {}` | 68.64 (+1.90) | 74.00 (+2.48) |
| 80-template ensemble | 69.50 (+2.76) | 74.60 (+3.08) |

One token — `a` — is worth 3.3 points on both models, more than the choice of noun (`image` vs `photo`).
So the notebook carries three variants: one that loses, one that wins, and the minimal-pair between them.

### 1. Half of $\Delta M$ cannot matter, and this is exact

55–58 % of $\|\Delta z_T\|^2$ is a **single direction shared by all 1000 classes**. Re-scoring with only
that shared part reproduces the bare accuracy to the last decimal; re-scoring with it removed reproduces
the templated accuracy to the last decimal (§1). The prompt's largest geometric effect is invisible to
`argmax`. Any account of prompt engineering that stops at "it moves the text embeddings" is describing
the part that provably does nothing.

### 2. It is never one component — on either side

* **Text side (§2).** The largest single component carries 4–5 % of $\Sigma\|\Delta b\|$; the top ten
  carry 30–36 %. Reaching 90 % of the class-varying change needs **71 of 109** components on ViT-B-16 and
  **102 of 157** on ViT-L-14. The late MLP slots `m9`–`m12` move most, in every variant and both models,
  with `a8.1`/`a7.9` the only attention heads in the top five.
* **Pair level (§3).** $\Delta\bar M$ is diffuse: the top 100 of 17 113 pairs hold 26 % of $|\Delta\bar M|$
  on ViT-B-16, and the top 100 of 64 213 hold 15–17 % on ViT-L-14. The discriminative part $G$ is barely
  more concentrated (31–33 % / 17–19 %).

So: **not one component activating, and not a gradual ramp over a few — a broad, low-amplitude
redistribution over nearly the whole text tower**, with the late MLP scaffold carrying the largest single
share.

### 3. The top interactions do not change; their weights do

In the (cos, weight) plane (§4) the vision-side weight $\|\hat a\|/\|z_I\|$ is **identical to 4 decimals**
across prompts — it must be, the vision tower never ran again — so every vision head moves purely
horizontally. Spearman correlation of the $\varphi$ ranking stays 0.73–0.92 and **13–15 of the top 15
heads survive on positive pairs**. The prompt reweights the same interactions rather than recruiting new
ones.

Two asymmetries are worth keeping:
* **Negative pairs reorder far more than positive ones** (top-15 overlap 5–12 vs 11–15). The template's
  effect on wrong classes is the less stable half.
* The direction change is much larger on the **text** side than the vision side. Biggest movers:
  `a8.5` (+0.107 Δcos), `a8.1`, `a8.3` on ViT-B-16; `a10.8` (+0.121…+0.147) on ViT-L-14 — mid-to-late
  text attention heads, consistently the same ones across all three templates.

### 4. What actually drives the accuracy: `m12` is the whole story, negatively

The causal swap (§5) is unambiguous. Swapping **exactly one** text component, the final MLP slot `m12`,
costs **−4.24 / −3.10 / −3.06** points on ViT-B-16 and **−2.98 / −2.16 / −2.52** on ViT-L-14 — for the
losing template *and both winning ones*. Nothing else comes within a factor of four of it.

The templates that win do **not** win by activating something. Other components contribute a few tenths
each (`m9` +0.80, `m7` +0.44, `a9.5`, `a7.5`, `a7.3` under `a photo of a {}`), and the sum of all
single-component effects is **−7.06** on ViT-L-14 while the true joint effect is **+2.48**.

The effect is strongly **non-additive**. No component, and no small set of components, explains the gain;
the gain exists only when the components are recombined together. That is the same lesson as §14 of
`playground_interaction_matrix.ipynb` — the coherent sum is the signal, individual components are not.
§8 makes the non-additivity of `m12` specifically into a measured quantity, and shows that `m12` is
**not** a nuisance term: it is the single most load-bearing text component even with no prompt involved.

### 5. Harm is concentrated, benefit is spread

§5c restricts $\Delta z_T$ to the top-$K$ vision components. Read the curves as a diagnostic, **not** as
an interpolation: the per-component terms sum to 1.6–11.5× less than their total absolute value
(the cancellation ratio), so partial sums dive far below *both* endpoints. That mutual cancellation is
itself the reason component-wise read-outs of $M$ keep failing.

The usable statistic is the share of $\Sigma_a|G_a|$ held by the top-10 vision components:

| | ViT-B-16 | ViT-L-14 |
|---|---|---|
| `an image of {}` | 63.0 % | 47.2 % |
| `an image of a {}` | 63.6 % | 42.4 % |
| `a photo of a {}` | **51.2 %** | **29.3 %** |

The winning template spreads its effect over the widest set of vision components on both models. And on
ViT-B-16 the contrast is visible directly in the curves: with only the single top component (`a11.5`)
connected, `an image of {}` has **already delivered its full 2-point loss** (leftmost blue point sits on
the templated line), whereas both winning templates sit on the *bare* line there and reach their gain
only once essentially the whole tower participates. On ViT-L-14 the K=1 point is below both endpoints for
every template, so that particular reading is ViT-B-16-specific; the top-10 share is the claim that holds
on both.

### 6. "Closer to the images" is the wrong statistic

The last question, measured four ways (§6):

* `an image of {}` moves the text embeddings **away** from the image cloud: mean cosine over all pairs
  drops 0.090 → 0.070 (B-16), 0.099 → 0.083 (L-14). Positive-pair cosine drops too.
* `a photo of a {}` leaves the mean cosine slightly **lower** than bare (0.090 → 0.087) yet gains
  1.90 points.

So mean closeness does not track accuracy, and neither does positive-pair cosine. The one statistic that
tracks it **monotonically across all 8 configurations** is the **margin against the best wrong class**
(0.0138 < 0.0164 < 0.0166 < 0.0179 on B-16, matching 64.74 < 66.74 < 68.02 < 68.64). Templating also makes
the class embeddings markedly more similar to *each other* (text–text cosine 0.508 → 0.58–0.61 for the
`a`-variants) without hurting — a per-class scaffold bias of exactly the kind
`playground_interaction_matrix.ipynb` §12 removes with Sinkhorn.

### Where this points

`m12`, the final text MLP, is the prompt-sensitive bottleneck: it is the largest mover in §2, the
dominant single-swap effect in §5 for every template, and a recurring partner in the top $G_{ab}$ pairs.
It is also, from the interaction-matrix work, the DC/scaffold component. The natural follow-up is whether
its class-shared component can be intervened on directly — which would be the training-free version of
prompt engineering, and connects to the Sinkhorn scaffold-bias result on the aggregate score.

## §8 — What exactly is `m12`, in what sense is it "the most contributing", and is it removable?

**Identity.** The text mlp slots are `m0 … m12`, where `m0` is the projected token+positional embedding
at the EOS position and `m1 … m12` are the MLP outputs of blocks 0 … 11. Both towers have 12 blocks, so
in **both** models **`m12` is the MLP of the last text block**, read at EOS, carried through its share of
`ln_final` and the text projection.

**"Most contributing" needs a test, and §5 only ran one.** The swap-in test moves one component while
the other 108/156 stay bare. Because the effect is strongly non-additive, that is *not* a decomposition
of the total. So this section adds two independent tests and a control:

| test | what it does |
|---|---|
| **A** swap-in (§5) | bare + $\Delta\mathbf b_b$ — the effect of the component's change **in isolation** |
| **B** leave-one-out | templated − $\Delta\mathbf b_b$ — the effect of removing it **in context** |
| **C** control | mean-ablate component $b$ of the **bare** decomposition — no prompt anywhere |

C is the one that matters for interpretation: mean-ablation replaces $\mathbf b_b(c)$ by
$\mathbb E_c[\mathbf b_b(c)]$, deleting exactly that component's class information (the constant left
behind is argmax-invisible, §1). If `m12` tops C as well, its dominance in A and B is a statement about
how much class signal it carries, not about prompts.

Then three candidate **interventions**, all built on the hypothesis that `m12` is a removable nuisance.

In [15]:
# Goal:  §8 -- three independent tests that "m12 dominates", plus the interventions it suggests.
@torch.no_grad()
def acc_zt(d, zT):
    return 100.0 * (d["lbl_t"][(d["A"].sum(1) @ zT.T).argmax(1).cpu().numpy()] == d["lbl_v"]).mean()

loo_rows, ctl_rows, int_rows = [], [], []
for m in MODELS:
    d = D[m]
    B0 = d["B"]["bare"]; z0 = B0.sum(1); i12 = list(d["nt"]).index("m12")
    a_bare = acc_zt(d, z0)
    # C -- control: delete each component's class information, no prompt involved
    for b in range(B0.shape[1]):
        z = z0 - B0[:, b] + B0[:, b].mean(0, keepdim=True)
        ctl_rows.append(dict(model=m, comp=d["nt"][b], d_acc=acc_zt(d, z) - a_bare))
    for p in VARIANTS:
        B1 = d["B"][p]; z1 = B1.sum(1); dB = B1 - B0
        a_tmpl = acc_zt(d, z1)
        # B -- leave-one-out from the templated end
        for b in range(dB.shape[1]):
            loo_rows.append(dict(model=m, prompt=p, comp=d["nt"][b],
                                 d_acc_loo=acc_zt(d, z1 - dB[:, b]) - a_tmpl,
                                 dnorm=float(dB[:, b].norm(dim=1).mean())))
        int_rows.append(dict(
            model=m, prompt=p, bare=a_bare, templated=a_tmpl,
            revert_m12=acc_zt(d, z1 - dB[:, i12]),                                  # template all BUT m12
            meanabl_m12_tmpl=acc_zt(d, z1 - B1[:, i12] + B1[:, i12].mean(0, keepdim=True)),
            meanabl_m12_bare=acc_zt(d, z0 - B0[:, i12] + B0[:, i12].mean(0, keepdim=True))))

df_loo = pd.DataFrame(loo_rows); df_ctl = pd.DataFrame(ctl_rows); df_int = pd.DataFrame(int_rows)
df_loo["per_norm"] = df_loo.d_acc_loo.abs() / df_loo.dnorm.clip(1e-9)

print("=== TEST B -- leave-one-out from templated (revert ONE component to bare), top 5 by |Δ|")
for (m, p), g in df_loo.groupby(["model", "prompt"], sort=False):
    g = g.reindex(g.d_acc_loo.abs().sort_values(ascending=False).index).head(5)
    r1, r2 = g.iloc[0], g.iloc[1]
    print(f"{m:9s} {p:17s} " + ", ".join(f"{r.comp} {r.d_acc_loo:+.2f}" for r in g.itertuples())
          + f"   [1st/2nd = {abs(r1.d_acc_loo)/max(abs(r2.d_acc_loo), 1e-9):.1f}x]")
print("\n=== per unit of movement: |Δacc_LOO| / E_c‖Δb_b‖, top 3")
for (m, p), g in df_loo.groupby(["model", "prompt"], sort=False):
    g = g.sort_values("per_norm", ascending=False).head(3)
    print(f"{m:9s} {p:17s} " + ", ".join(f"{r.comp} {r.per_norm:.0f}" for r in g.itertuples()))
print("\n=== TEST C -- CONTROL: mean-ablate one component of the BARE decomposition (no prompt), top 6")
for m, g in df_ctl.groupby("model", sort=False):
    g = g.reindex(g.d_acc.abs().sort_values(ascending=False).index).head(6)
    print(f"{m:9s} " + ", ".join(f"{r.comp} {r.d_acc:+.2f}" for r in g.itertuples()))
print("\n=== INTERVENTIONS  (every one of them LOSES -- m12 is load-bearing, not a nuisance)")
print(df_int.set_index(["model", "prompt"]).round(2).to_string())
for df, n in ((df_loo, "m12_loo"), (df_ctl, "m12_control"), (df_int, "m12_interventions")):
    df.to_csv(f"{FIG_DIR}/{n}.csv", index=False)
RESULTS["m12_interventions"] = df_int.to_dict("records")
RESULTS["m12_control_top"] = df_ctl.reindex(df_ctl.d_acc.abs().sort_values(ascending=False).index
                                            ).head(12).to_dict("records")

=== TEST B -- leave-one-out from templated (revert ONE component to bare), top 5 by |Δ|
ViT-B-16  an image of {}    m12 -3.68, a11.7 -1.24, m11 +1.06, a7.7 -0.80, a9.6 -0.78   [1st/2nd = 3.0x]
ViT-B-16  an image of a {}  m12 -9.08, a11.7 -1.22, m11 -0.74, a9.6 -0.58, a7.7 -0.50   [1st/2nd = 7.4x]
ViT-B-16  a photo of a {}   m12 -10.12, a11.7 -1.16, m9 -0.84, m11 -0.68, a7.7 -0.48   [1st/2nd = 8.7x]
ViT-L-14  an image of {}    m12 -9.72, m11 +1.12, m10 +1.12, a10.4 -0.52, m9 +0.34   [1st/2nd = 8.7x]
ViT-L-14  an image of a {}  m12 -13.94, a10.4 -0.94, a11.4 -0.66, a10.0 -0.60, m10 -0.52   [1st/2nd = 14.8x]
ViT-L-14  a photo of a {}   m12 -12.68, m11 -0.84, a11.4 -0.52, a10.0 -0.50, m10 -0.46   [1st/2nd = 15.1x]

=== per unit of movement: |Δacc_LOO| / E_c‖Δb_b‖, top 3
ViT-B-16  an image of {}    a11.7 51, a7.7 51, a8.0 36
ViT-B-16  an image of a {}  m12 87, a11.7 52, a7.7 30
ViT-B-16  a photo of a {}   m12 99, a11.7 50, a5.4 33
ViT-L-14  an image of {}    m12 61, a4.8 30, a8.9 29
ViT-L-1

### §8.1 Reading — the earlier framing was wrong in one respect

All three tests agree that `m12` dominates, and test **B** separates it far more sharply than the
swap-in test did: reverting `m12` alone from the fully-templated embedding costs **−3.7 to −14.0**
points, **3.0× to 15.1×** the next-largest component. Per unit of movement it is also first in 5 of the
6 configurations.

But the **control settles the interpretation**: with no prompt anywhere, mean-ablating `m12` in the bare
decomposition costs **−8.00** (ViT-B-16) and **−5.18** (ViT-L-14) — the largest of any component on both
models. So `m12` is the single most **load-bearing** text component for zero-shot in general. Its
dominance under prompting follows from that; it is not evidence that the template "breaks" it.

Consequently all three interventions **fail**, and they fail badly:

| | bare | templated | revert `m12` | mean-abl `m12` (tmpl) | mean-abl `m12` (bare) |
|---|---|---|---|---|---|
| B-16 `a photo of a {}` | 66.74 | 68.64 | 58.52 | 62.36 | 58.74 |
| L-14 `a photo of a {}` | 71.52 | 74.00 | 61.32 | 70.22 | 66.34 |

`m12` cannot be removed, reverted, or mean-ablated — every variant lands 5–10 points below both endpoints.

The quantity that *is* interesting is the **gap between tests A and B for the same component**. For
`a photo of a {}` on ViT-B-16, swapping `m12` in alone is worth **−3.06**, while removing it from the
templated embedding is worth **−10.12**. `m12`'s templated value is therefore useful *only in the
context of the other components' templated values*: a ~13-point interaction term localised on a single
component. That is the non-additivity of §5, measured rather than asserted.

## §9 — Top-down: positives or negatives, and how many components? (falsifiable at every level)

Zero-shot top-1 is $\arg\max_c s(i,c)$, so the gain must show up in the *margin*
$s(i,y_i)-\max_{c\neq y_i}s(i,c)$. Four levels, each an exact identity, each with a hypothesis that the
numbers can kill. The losing template `an image of {}` is the **sign control**: a real mechanism must
explain it with the opposite sign.

| level | test | hypotheses it can falsify |
|---|---|---|
| **L1** | replace only the true-class entry (`pos-only`) or only the wrong-class entries (`neg-only`) with the templated value | **H1** positive-driven → `pos-only` ≈ templated · **H2** negative-driven → converse · **H3** both → gains add |
| **L2** | score statistics: true class, the incumbent competitor $c^*$ (fixed from the **bare** scores, never post-hoc), the running $\max$ over wrong classes, their spread, the true class's rank | **H2b** "it suppresses the top competitor" → that statistic should separate winners from the loser |
| **L3** | $\Phi_b=\mathbb E_i[\langle z_I,\Delta\mathbf b_b(y_i)\rangle-\langle z_I,\Delta\mathbf b_b(c^*_i)\rangle]$, an exact additive split of the margin change over text components, itself split into a positive and a negative half | **H4** few components → $\Phi$ concentrated |
| **L4** | swap the top-$\Phi$ components in first and watch the accuracy | **H4c** the concentrated components are *causally sufficient* |

All hybrids use the **centred** change $\Delta\tilde s(i,c)=\Delta s(i,c)-\mathbb E_c[\Delta s(i,c)]$, since
§1 showed the class-shared part is argmax-invisible; adding it to one entry only would inject a spurious
bias. `pos-only` + `neg-only` − bare reproduces the templated argmax exactly (asserted in the cell).

In [16]:
# Goal:  §9 -- L1..L4. Every level is an identity, so each hypothesis stands or falls on its own numbers.
t1, t2, t3, t5 = [], [], [], []
for m in MODELS:
    d = D[m]
    zI = d["A"].sum(1); lbl = torch.as_tensor(d["lbl_v"].astype(np.int64), device=device)
    B0 = d["B"]["bare"]; z0 = B0.sum(1); S0 = zI @ z0.T
    N, Cn = S0.shape; ar = torch.arange(N, device=device)
    S0m = S0.clone(); S0m[ar, lbl] = -1e9
    cstar = S0m.argmax(1)                       # incumbent competitor, fixed from the BARE scores
    onehot = torch.zeros(N, Cn, dtype=torch.bool, device=device); onehot[ar, lbl] = True
    lab_np = d["lbl_v"]
    accS = lambda S: 100.0 * (S.argmax(1) == lbl).float().mean().item()
    a0 = accS(S0)
    for p in VARIANTS:
        B1 = d["B"][p]; dB = B1 - B0; z1 = B1.sum(1); S1 = zI @ z1.T; a1 = accS(S1)
        dS = S1 - S0; dSc = dS - dS.mean(1, keepdim=True)
        assert abs(accS(S0 + dSc) - a1) < 1e-9, "centring must preserve the argmax"
        # ---- L1
        ap, an = accS(S0 + dSc * onehot), accS(S0 + dSc * ~onehot)
        t1.append(dict(model=m, prompt=p, bare=a0, templated=a1, gain=a1 - a0,
                       pos_only=ap, neg_only=an, gain_pos=ap - a0, gain_neg=an - a0,
                       sum_of_parts=(ap - a0) + (an - a0)))
        # ---- L2
        def stats(S):
            Sc = S - S.mean(1, keepdim=True); sp = Sc[ar, lbl]
            Sw = Sc.clone(); Sw[ar, lbl] = -1e9
            return dict(s_pos=sp.mean().item(), s_cstar=Sc[ar, cstar].mean().item(),
                        s_maxneg=Sw.max(1).values.mean().item(), s_negstd=Sc[~onehot].std().item(),
                        margin=(sp - Sw.max(1).values).mean().item(),
                        margin_fixed=(sp - Sc[ar, cstar]).mean().item(),
                        mean_rank=(S > S[ar, lbl][:, None]).sum(1).float().mean().item())
        bb, tt = stats(S0), stats(S1)
        S1m = S1.clone(); S1m[ar, lbl] = -1e9
        t2.append(dict(model=m, prompt=p, gain=a1 - a0,
                       competitor_changed_pct=100.0 * (S1m.argmax(1) != cstar).float().mean().item(),
                       **{f"d_{k}": tt[k] - bb[k] for k in bb}))
        # ---- L3  (dbar cancels in Phi; it is subtracted from P and Nn so each half is interpretable)
        dbar = dB.mean(0)
        P = torch.zeros(dB.shape[1], dtype=torch.float64, device=device)
        Nn = torch.zeros_like(P); Sh = torch.zeros_like(P)
        for i0 in range(0, N, 512):
            sl = slice(i0, min(i0 + 512, N)); zi = zI[sl]
            P  += torch.einsum("ibd,id->b", dB[lbl[sl]],   zi).double()
            Nn += torch.einsum("ibd,id->b", dB[cstar[sl]], zi).double()
            Sh += (zi @ dbar.T).sum(0).double()
        P, Nn, Sh = P / N, Nn / N, Sh / N
        Phi = P - Nn; Pc, Nc = P - Sh, Nn - Sh
        truth = (dS[ar, lbl] - dS[ar, cstar]).mean().item()
        assert abs(float(Phi.sum()) - truth) < 1e-6, "Phi must sum to the margin change"
        o = np.argsort(-Phi.abs().cpu().numpy())
        cum = np.cumsum(Phi.cpu().numpy()[o]) / float(Phi.sum())
        av = Phi.abs()
        t3.append(dict(model=m, prompt=p, sum_Phi=float(Phi.sum()), identity_err=abs(float(Phi.sum()) - truth),
                       pos_part=float(Pc.sum()), neg_part=float(-Nc.sum()),
                       neg_share_pct=100 * float(-Nc.sum()) / float(Phi.sum()),
                       top1=100 * float(av[o[0]] / av.sum()), top5=100 * float(av[o[:5]].sum() / av.sum()),
                       top10=100 * float(av[o[:10]].sum() / av.sum()),
                       k_for_80pct=int(np.searchsorted(cum, 0.8) + 1), Kt=len(Phi)))
        print(f"{m:9s} '{p:16s}' top Phi_b: " + ", ".join(
            f"{d['nt'][i]} {Phi[i]:+.5f}(pos{Pc[i]:+.5f}/neg{-Nc[i]:+.5f})" for i in o[:5]))
        # ---- L4
        ks = np.unique(np.r_[0, np.round(np.geomspace(1, len(Phi), 14)).astype(int)])
        for oname, order in (("by Phi_b", o), ("by ‖Δb‖", np.argsort(-dB.norm(dim=2).mean(0).cpu().numpy())),
                             ("random", np.random.default_rng(0).permutation(len(Phi)))):
            z = z0.clone(); prev = 0
            for k in ks:
                if k > prev:
                    sel = torch.as_tensor(np.ascontiguousarray(order[prev:k]).astype(np.int64), device=device)
                    z = z + dB[:, sel].sum(1); prev = k
                t5.append(dict(model=m, prompt=p, order=oname, k=int(k), acc=accS(zI @ z.T)))

df_l1, df_l2, df_l3 = pd.DataFrame(t1), pd.DataFrame(t2), pd.DataFrame(t3)
df_l5 = pd.DataFrame(t5)
print("\n===== L1  which column carries it  (gain_pos + gain_neg vs the real gain)")
print(df_l1.set_index(["model", "prompt"]).round(2).to_string())
print("\n===== L2  score statistics, centred.  c* = incumbent competitor, fixed from the bare scores")
print(df_l2.set_index(["model", "prompt"]).round(4).to_string())
print("\n===== L3  exact additive split of the margin change over text components")
print(df_l3.set_index(["model", "prompt"]).round(4).to_string())
print("\n===== L4  accuracy vs #components swapped in, by ordering")
print(df_l5.pivot_table(index=["model", "prompt", "order"], columns="k", values="acc").round(2).to_string())
for df, n in ((df_l1, "L1_pos_vs_neg"), (df_l2, "L2_score_stats"), (df_l3, "L3_phi"), (df_l5, "L4_causal")):
    df.to_csv(f"{FIG_DIR}/{n}.csv", index=False)
RESULTS["L1"] = df_l1.to_dict("records"); RESULTS["L2"] = df_l2.to_dict("records")
RESULTS["L3"] = df_l3.to_dict("records")

ViT-B-16  'an image of {}  ' top Phi_b: m12 +0.00292(pos+0.00011/neg+0.00281), m9 +0.00207(pos-0.00058/neg+0.00265), m11 +0.00099(pos-0.00000/neg+0.00100), m10 +0.00082(pos-0.00087/neg+0.00169), m5 +0.00034(pos+0.00068/neg-0.00033)


ViT-B-16  'an image of a {}' top Phi_b: m12 +0.00318(pos+0.00208/neg+0.00110), m9 +0.00211(pos-0.00016/neg+0.00227), m11 +0.00111(pos+0.00033/neg+0.00078), m10 +0.00083(pos-0.00071/neg+0.00154), m5 +0.00035(pos+0.00093/neg-0.00058)


ViT-B-16  'a photo of a {} ' top Phi_b: m12 +0.00341(pos-0.00006/neg+0.00347), m9 +0.00232(pos+0.00094/neg+0.00138), m11 +0.00114(pos+0.00004/neg+0.00110), m10 +0.00084(pos+0.00022/neg+0.00062), m7 +0.00046(pos+0.00015/neg+0.00031)


ViT-L-14  'an image of {}  ' top Phi_b: m12 +0.01022(pos+0.00228/neg+0.00794), m11 +0.00286(pos+0.00020/neg+0.00265), m10 +0.00263(pos+0.00006/neg+0.00256), m9 +0.00142(pos+0.00052/neg+0.00089), m8 +0.00113(pos+0.00028/neg+0.00085)


ViT-L-14  'an image of a {}' top Phi_b: m12 +0.01169(pos+0.00430/neg+0.00740), m11 +0.00324(pos+0.00041/neg+0.00283), m10 +0.00180(pos-0.00003/neg+0.00182), m9 +0.00142(pos+0.00005/neg+0.00137), m8 +0.00094(pos+0.00038/neg+0.00056)


ViT-L-14  'a photo of a {} ' top Phi_b: m12 +0.01141(pos+0.00202/neg+0.00939), m11 +0.00346(pos+0.00043/neg+0.00303), m10 +0.00175(pos+0.00055/neg+0.00120), m9 +0.00154(pos+0.00016/neg+0.00138), m8 +0.00089(pos+0.00022/neg+0.00067)



===== L1  which column carries it  (gain_pos + gain_neg vs the real gain)
                            bare  templated  gain  pos_only  neg_only  gain_pos  gain_neg  sum_of_parts
model    prompt                                                                                        
ViT-B-16 an image of {}    66.74      64.74 -2.00     65.20     65.16     -1.54     -1.58         -3.12
         an image of a {}  66.74      68.02  1.28     67.80     66.20      1.06     -0.54          0.52
         a photo of a {}   66.74      68.64  1.90     70.26     64.14      3.52     -2.60          0.92
ViT-L-14 an image of {}    71.52      68.98 -2.54     65.76     74.14     -5.76      2.62         -3.14
         an image of a {}  71.52      73.46  1.94     68.96     74.78     -2.56      3.26          0.70
         a photo of a {}   71.52      74.00  2.48     73.62     71.14      2.10     -0.38          1.72

===== L2  score statistics, centred.  c* = incumbent competitor, fixed from the bare scores


In [17]:
# Goal:  §9 -- the figure: L2 statistics against the accuracy change, and the L4 causal curves.
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
ORD = list(VARIANTS)
# top row: which score statistic tracks the accuracy change?
for ax, (k, ttl) in zip(axes[0], [("d_margin_fixed", "Δ margin vs the INCUMBENT c*\n(positive even for the loser)"),
                                  ("d_margin", "Δ margin vs the RUNNING max\n(tracks the accuracy, sign and order)"),
                                  ("d_mean_rank", "Δ mean rank of the true class\n(lower = better)")]):
    for j, p in enumerate(ORD):
        g = df_l2[df_l2.prompt == p]
        ax.scatter(g.gain, g[k], s=70, c=P_COLOR[p], marker=P_MARKER[p], lw=0, label=p, zorder=3)
        for _, r in g.iterrows():
            ax.annotate(r.model.replace("ViT-", ""), (r.gain, r[k]), fontsize=6,
                        xytext=(5, -2), textcoords="offset points")
    ax.axhline(0, color="#8a8a86", lw=0.8); ax.axvline(0, color="#8a8a86", lw=0.8)
    ax.set_xlabel("Δ zero-shot top-1 [%]"); ax.set_title(ttl, fontsize=8.5); ax.tick_params(labelsize=7)
axes[0][0].legend(fontsize=7, frameon=False, title="prompt", title_fontsize=7)
# bottom row: L4 causal curves, one panel per model + the Phi concentration
for ax, m in zip(axes[1][:2], MODELS):
    for p in ORD:
        for oname, ls in (("by Phi_b", "-"), ("random", ":")):
            g = df_l5[(df_l5.model == m) & (df_l5.prompt == p) & (df_l5.order == oname)].sort_values("k")
            ax.plot(g.k, g.acc, color=P_COLOR[p], ls=ls, lw=1.6 if ls == "-" else 1.0)
    ax.axhline(df_l1[df_l1.model == m].bare.iloc[0], color="#c3c2b7", lw=0.9)
    ax.set_title(f"{m} — swap in top-Φ components first (solid) vs random (dotted)", fontsize=8.5)
    ax.set_xlabel("number of text components swapped in"); ax.set_ylabel("zero-shot top-1 [%]")
    ax.tick_params(labelsize=7)
ax = axes[1][2]
w = 0.26
for j, p in enumerate(ORD):
    g = df_l3[df_l3.prompt == p]
    b = ax.bar(np.arange(len(MODELS)) + (j - 1) * w, g.top5.values, w * 0.82, color=P_COLOR[p], lw=0, label=p)
    ax.bar_label(b, fmt="%.0f%%", fontsize=6.5, padding=2)
ax.set_xticks(np.arange(len(MODELS))); ax.set_xticklabels(MODELS, fontsize=7.5)
ax.set_title("share of |Φ| in the top-5 text components\n(the margin IS concentrated)", fontsize=8.5)
ax.set_ylabel("[%]"); ax.grid(axis="x", alpha=0); ax.tick_params(labelsize=7)
fig.suptitle("§9  The margin change is concentrated and negative-side; the accuracy change is neither",
             fontsize=11, y=1.02)
plt.tight_layout(); save_fig(fig, "09_posneg_topdown"); plt.show()

saved -> output_dir/activations_and_datasets_idxs_69/prompt_interaction_figs/09_posneg_topdown.png
